In [ ]:
import zipfile,os,json,random,re
import pandas as pd
import torch
import numpy as np

In [ ]:
from datasets import load_dataset
dataset = load_dataset("BothBosu/scam-dialogue")

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()


README.md:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

scam-dialogue_train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

scam-dialogue_test.csv:   0%|          | 0.00/453k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1280 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/320 [00:00<?, ? examples/s]

In [ ]:
train_df = train_df.rename(columns={"dialogue": "dialogue_text"})
test_df = test_df.rename(columns={"dialogue": "dialogue_text"})

train_df = train_df[["dialogue_text","label"]]
test_df = test_df[["dialogue_text","label"]]

print("Train distribution:",train_df["label"].value_counts().to_dict())
print("Test distribution:",test_df["label"].value_counts().to_dict())

Train distribution: {1: 640, 0: 640}
Test distribution: {1: 160, 0: 160}


In [ ]:
from sklearn.model_selection import train_test_split

if"test_df" not in dir() or test_df is None:
  train_df,test_df = train_test_split(
      train_df, test_size=0.2,
      startify=train_df["label"], random_state=42
  )
print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

label
1    640
0    640
Name: count, dtype: int64
label
1    160
0    160
Name: count, dtype: int64


In [ ]:
def split_turns_flexible(text):
  """
    Split dialogue into turns.
    If 4c prefixes exist (Suspect:, Innocent:, Agent:, etc.), split on those.
    Otherwise, split on sentence boundaries as fallback.
  """
  if not isinstance(text,str) or not text.strip():
    return []

  #speaker-based splitting
  speaker_pattern = r"(?=\b(?:caller|receiver|agent|customer|scammer|victim|suspect|innocent)\s*:)"
  if re.search(speaker_pattern,text,flags=re.IGNORECASE):
    parts = re.split(speaker_pattern,text,flags=re.IGNORECASE)
    return [p.strip() for p in parts if p.strip()]

  # Fallback: split on sentence boundaries
  sentences = re.split(r'(?<=[.!?])\s+', text.strip())
  return [s.strip() for s in sentences if s.strip()]

In [ ]:
# Apply to both splits
train_df["turns"] = train_df["dialogue_text"].apply(split_turns_flexible)
test_df["turns"] = test_df["dialogue_text"].apply(split_turns_flexible)

train_df["n_turns"] = train_df["turns"].apply(len)
test_df["n_turns"] = test_df["turns"].apply(len)

# Remove any rows that produced zero turns
train_df = train_df[train_df["n_turns"] > 0].reset_index(drop=True)
test_df = test_df[test_df["n_turns"] > 0].reset_index(drop=True)

print(train_df["n_turns"].describe())

count    1280.000000
mean       13.657813
std         4.332191
min         6.000000
25%        10.000000
50%        13.000000
75%        17.000000
max        28.000000
Name: n_turns, dtype: float64


In [ ]:
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
bert_model = bert_model.to(device).eval()

def embed_turns_batch(turns_list, batch_size=64, max_length=64):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(turns_list), batch_size):
            batch = turns_list[i:i+batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
            cls_emb = bert_model(**enc).last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0) if all_embeddings else torch.empty(0, 768)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def flatten_turns(df):
    flat_turns, dialogue_ids = [], []
    for idx, turns in enumerate(df["turns"]):
        for t in turns:
            flat_turns.append(t)
            dialogue_ids.append(idx)
    return flat_turns, dialogue_ids

In [ ]:
def get_dialogue_embeddings(df):
    flat_turns, dialogue_ids = flatten_turns(df)
    turn_embeddings = embed_turns_batch(flat_turns)
    grouped = defaultdict(list)
    for emb, d_id in zip(turn_embeddings, dialogue_ids):
        grouped[d_id].append(emb)
    return [torch.stack(grouped[i]) if i in grouped else torch.zeros(1, 768) for i in range(len(df))]

In [ ]:

train_embeddings = get_dialogue_embeddings(train_df)
test_embeddings = get_dialogue_embeddings(test_df)
print(len(train_embeddings), train_embeddings[0].shape)

1280 torch.Size([13, 768])


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class DialogueDataset(Dataset):
    def __init__(self, embeddings_list, labels):
        self.embeddings, self.labels = embeddings_list, labels
    def __len__(self): return len(self.embeddings)
    def __getitem__(self, idx): return self.embeddings[idx], self.labels[idx]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)
    return padded, lengths, torch.tensor(labels, dtype=torch.float32)

train_loader = DataLoader(DialogueDataset(train_embeddings, train_df["label"].tolist()),
                           batch_size=16, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(DialogueDataset(test_embeddings, test_df["label"].tolist()),
                          batch_size=16, shuffle=False, collate_fn=collate_fn)


In [ ]:
class BiLSTMFraudDetector(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128):
        super().__init__()
        self.bilstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.bilstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, out.size(2)).to(out.device)
        last_hidden = out.gather(1, idx).squeeze(1)
        return self.classifier(last_hidden).squeeze(-1)


In [ ]:
model = BiLSTMFraudDetector().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(3):
    model.train()
    total_loss = 0
    for x, lengths, labels in train_loader:
        x, labels = x.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, lengths), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, loss: {total_loss/len(train_loader):.4f}")

Epoch 1, loss: 0.6417
Epoch 2, loss: 0.2604
Epoch 3, loss: 0.0709


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, lengths, labels in test_loader:
        probs = torch.sigmoid(model(x.to(device), lengths)).cpu()
        all_preds.extend((probs >= 0.5).int().tolist())
        all_labels.extend(labels.int().tolist())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

Accuracy: 0.971875
F1: 0.9724770642201835
[[152   8]
 [  1 159]]
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       160
           1       0.95      0.99      0.97       160

    accuracy                           0.97       320
   macro avg       0.97      0.97      0.97       320
weighted avg       0.97      0.97      0.97       320



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Concatenate HF train text
X_train = train_df["dialogue_text"].tolist()
y_train = train_df["label"].tolist()
X_test = test_df["dialogue_text"].tolist()
y_test = test_df["label"].tolist()

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
print("TF-IDF + LR accuracy:", pipe.score(X_test, y_test))

TF-IDF + LR accuracy: 1.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class CausalLSTM(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        return self.head(out).squeeze(-1)          # one score per turn

    def step(self, turn_emb, state=None):          # used by the web app, one turn at a time
        out, state = self.lstm(turn_emb, state)
        return torch.sigmoid(self.head(out)).view(-1), state

In [ ]:
torch.manual_seed(42)
causal_model = CausalLSTM().to(device)
opt = torch.optim.Adam(causal_model.parameters(), lr=1e-4)
bce = nn.BCEWithLogitsLoss(reduction="none")

for epoch in range(3):
    causal_model.train()
    total = 0
    for x, lengths, labels in train_loader:
        x, labels = x.to(device), labels.to(device)
        logits = causal_model(x, lengths)                       # [batch, turns]
        mask = (torch.arange(logits.size(1), device=device)[None, :] < lengths.to(device)[:, None]).float()
        target = labels[:, None].expand_as(logits)              # call label applied to every turn
        loss = (bce(logits, target) * mask).sum() / mask.sum()
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch+1}: loss {total/len(train_loader):.4f}")

epoch 1: loss 0.6342
epoch 2: loss 0.3804
epoch 3: loss 0.2021


In [ ]:
import os
OUT = "/content/drive/MyDrive/fraud_detector"
os.makedirs(OUT, exist_ok=True)

torch.save(causal_model.state_dict(), f"{OUT}/causal_lstm.pt")   # 1. classifier weights
bert_model.save_pretrained(f"{OUT}/distilbert")                   # 2. embedder
tokenizer.save_pretrained(f"{OUT}/distilbert")                    # 3. tokenizer (same folder)

print(os.listdir(OUT))
print(os.listdir(f"{OUT}/distilbert"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['causal_lstm.pt', 'distilbert']
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
check = CausalLSTM().to(device)
check.load_state_dict(torch.load(f"{OUT}/causal_lstm.pt", map_location=device))
print("Saved model loads correctly")

Saved model loads correctly
